# 🎬 vTranslate Studio - Cloud Pipeline (Google Colab GPU)
### Quy trình Độc lập: OCR ➔ Xuất file SRT về Drive ➔ Bạn tự dịch ➔ Colab Lồng tiếng TTS & Render Video
---
Notebook này được chia làm **2 Giai đoạn linh hoạt** (không sợ nghẽn hay hết hạn mức API key):
* **GIAI ĐOẠN 1 (Tách chữ OCR)**: Quét phụ đề video từ Google Drive ➔ Xuất tệp `video_goc.srt` lưu thẳng vào Google Drive. Bạn tải file SRT về tự dịch bằng bất kỳ công cụ nào bạn muốn.
* **GIAI ĐOẠN 2 (Lồng tiếng & Render)**: Đọc tệp `video_tieng_viet.srt` đã dịch trên Drive ➔ VieNeu-TTS (GPU) sinh giọng đọc `Gemini Puck` ➔ Render video lật gương ngang, làm mờ phụ đề cũ, chèn logo, đốt phụ đề ➔ Lưu video thành phẩm vào Google Drive.

*(Toàn bộ thông số: Vùng quét OCR/Blur `111, 1232, 819, 96`, Lật ngược ngang, Âm lượng 17%/100%, Giọng Puck, Logo góc trên phải... đều đã được **SET CỨNG CHUẨN XÁC** theo dự án của bạn)*.

In [ ]:
# @title 🚀 1. Cài đặt môi trường & Thư viện (Chạy 1 lần đầu)
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg fonts-liberation
!pip install -q rapidocr-onnxruntime vieneu soundfile numpy gdown

In [ ]:
# @title 📁 2. Kết nối Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Đã kết nối Google Drive thành công!")

In [ ]:
# @title ⚙️ 3. Thiết lập Đường dẫn & Cấu hình Set Cứng

# --- 1. THIẾT LẬP ĐƯỜNG DẪN TRÊN GOOGLE DRIVE ---
# Đường dẫn video nguồn trên Drive (hoặc dán Link chia sẻ Google Drive):
INPUT_VIDEO_SOURCE = "/content/drive/MyDrive/红果短剧/东北话事人-7eb94773ec59/东北话事人-7eb94773ec59_FULL.mp4" # @param {type:"string"}

# Thư mục trên Google Drive để lưu tệp SRT và Video thành phẩm:
OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/vTranslator_Outputs" # @param {type:"string"}

# Đường dẫn tệp Logo trên Drive (để trống nếu không chèn):
LOGO_SOURCE = "/content/drive/MyDrive/Biểu tượng S đỏ bóng và SHREWTV.png" # @param {type:"string"}

# Đường dẫn tệp SRT đã dịch tiếng Việt trên Drive (Dùng cho Giai đoạn 2):
# (Mặc định nó sẽ tự tìm file cùng tên video có đuôi _vi.srt trong thư mục OUTPUT_DRIVE_DIR)
TRANSLATED_SRT_PATH = "" # @param {type:"string"}

# --- 2. TOÀN BỘ CẤU HÌNH ĐÃ SET CỨNG THEO DỰ ÁN CỦA BẠN ---
HARDCODED_CONFIG = {
    "ocr_roi": {
        "x": 111,
        "y": 1232,
        "width": 819,
        "height": 96,
        "scene_width": 1080,
        "scene_height": 1920,
    },
    "flip_mode": "Horizontal",  # Lật gương ngang
    "zoom_percent": 100,
    "blur": {
        "enabled": True,
        "mode": "blur",
        "x": 111,
        "y": 1232,
        "width": 819,
        "height": 96,
        "blur_radius": 15,
        "opacity": 1.0,
    },
    "audio": {
        "original_volume": 0.17,    # 17% âm lượng video gốc
        "dubbing_volume": 1.0,     # 100% âm lượng lồng tiếng
        "max_tempo_stretch": 1.22, # Co giãn nhịp đọc tối đa 1.22x
        "voice_id": "Gemini Puck"   # Mẫu giọng VieNeu
    },
    "logo": {
        "enabled": True,
        "position": "top_right",
        "scale_percent": 12,
        "opacity": 0.85,
        "margin_x": 24,
        "margin_y": 24,
    },
    "subtitle_style": {
        "font_name": "Arial",
        "font_size": 34,
        "primary_color": "&H00FFFFFF", # Màu trắng
        "outline_color": "&H007F7B7B", # Viền xám tro
        "outline_width": 1,
        "margin_v": 39,
        "alignment": 2,
        "single_line": True
    }
}

os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)

import re
import subprocess

def resolve_drive_file(source_path, target_local_path):
    if not source_path:
        return ""
    if source_path.startswith("http://") or source_path.startswith("https://"):
        match = re.search(r'/d/([a-zA-Z0-9_-]+)', source_path) or re.search(r'id=([a-zA-Z0-9_-]+)', source_path)
        if match:
            fid = match.group(1)
            print(f"Phát hiện link Google Drive ID: {fid}. Đang tải...")
            subprocess.run(["gdown", "--id", fid, "-O", target_local_path], check=True)
            return target_local_path
        else:
            print(f"Đang tải file từ URL: {source_path}...")
            subprocess.run(["wget", "-O", target_local_path, source_path], check=True)
            return target_local_path
    return source_path

LOCAL_VIDEO_PATH = resolve_drive_file(INPUT_VIDEO_SOURCE, "/content/input_video.mp4")
LOCAL_LOGO_PATH = resolve_drive_file(LOGO_SOURCE, "/content/logo.png")

print(f"\n✅ Tệp video sẵn sàng: {LOCAL_VIDEO_PATH}")
print(f"✅ Thư mục xuất trên Google Drive: {OUTPUT_DRIVE_DIR}")

In [ ]:
# @title 📝 4. GIAI ĐOẠN 1: Quét OCR ➔ Xuất file .SRT (Lưu thẳng vào Google Drive)
import cv2
import time
from rapidocr_onnxruntime import RapidOCR

def ms_to_srt_time(ms):
    h = ms // 3600000
    m = (ms % 3600000) // 60000
    s = (ms % 60000) // 1000
    milli = ms % 1000
    return f"{h:02d}:{m:02d}:{s:02d},{milli:03d}"

def run_ocr_and_export_srt(video_path, roi_cfg, sample_interval=0.4):
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Không tìm thấy tệp video: {video_path}")
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step_frames = max(1, int(fps * sample_interval))
    
    ocr = RapidOCR(use_cls=False, det_db_score_mode="fast", det_limit_side_len=640)
    rx, ry, rw, rh = roi_cfg['x'], roi_cfg['y'], roi_cfg['width'], roi_cfg['height']
    sw, sh = roi_cfg.get('scene_width', 1080), roi_cfg.get('scene_height', 1920)
    
    cur_frame = 0
    active_text = ""
    active_start_sec = 0.0
    active_end_sec = 0.0
    segments = []
    seg_idx = 1
    t0 = time.time()
    last_reported = 0
    
    print(f"🔍 Bắt đầu quét phụ đề từ video ({total_frames} frames)...\n")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        t_sec = cur_frame / fps
        h, w = frame.shape[:2]
        
        sx, sy = w / sw, h / sh
        x1, y1 = max(0, min(w - 10, int(rx * sx))), max(0, min(h - 10, int(ry * sy)))
        x2, y2 = max(x1 + 10, min(w, int((rx + rw) * sx))), max(y1 + 10, min(h, int((ry + rh) * sy)))
        
        crop = frame[y1:y2, x1:x2]
        crop_pad = cv2.copyMakeBorder(crop, 16, 16, 16, 16, cv2.BORDER_CONSTANT, value=[0, 0, 0])
        res, _ = ocr(crop_pad, use_det=True, use_cls=False, use_rec=True)
        
        text = ""
        if res:
            res.sort(key=lambda x: x[0][0][0])
            texts = [str(item[1]).strip() for item in res if item[1] and float(item[2]) >= 0.45]
            text = re.sub(r'\s+', ' ', " ".join(texts)).strip()
            
        if text:
            if not active_text:
                active_text = text
                active_start_sec = t_sec
                active_end_sec = t_sec + sample_interval
            elif text == active_text:
                active_end_sec = t_sec + sample_interval
            else:
                if active_end_sec - active_start_sec >= 0.4:
                    segments.append({
                        "index": seg_idx,
                        "start_ms": int(active_start_sec * 1000),
                        "end_ms": int(active_end_sec * 1000),
                        "text": active_text
                    })
                    seg_idx += 1
                active_text = text
                active_start_sec = t_sec
                active_end_sec = t_sec + sample_interval
        else:
            if active_text:
                if active_end_sec - active_start_sec >= 0.4:
                    segments.append({
                        "index": seg_idx,
                        "start_ms": int(active_start_sec * 1000),
                        "end_ms": int(active_end_sec * 1000),
                        "text": active_text
                    })
                    seg_idx += 1
                active_text = ""
                
        for _ in range(step_frames - 1):
            if not cap.grab():
                break
        cur_frame += step_frames
        
        if total_frames > 0:
            pct = int((cur_frame / total_frames) * 100)
            if pct >= last_reported + 10:
                last_reported = pct
                print(f"⏳ Tiến độ OCR: {pct}% | Đã bắt được: {len(segments)} câu")
                
    cap.release()
    if active_text and (active_end_sec - active_start_sec >= 0.4):
        segments.append({
            "index": seg_idx,
            "start_ms": int(active_start_sec * 1000),
            "end_ms": int(active_end_sec * 1000),
            "text": active_text
        })
        
    # Xuất file SRT vào Google Drive
    base_name = os.path.splitext(os.path.basename(INPUT_VIDEO_SOURCE))[0]
    raw_srt_path = os.path.join(OUTPUT_DRIVE_DIR, f"{base_name}_goc.srt")
    
    with open(raw_srt_path, "w", encoding="utf-8") as f:
        for s in segments:
            s_str = ms_to_srt_time(s["start_ms"])
            e_str = ms_to_srt_time(s["end_ms"])
            f.write(f"{s['index']}\n{s_str} --> {e_str}\n{s['text']}\n\n")
            
    print(f"\n🎉🎉 Xong OCR trong {time.time() - t0:.1f}s!")
    print(f"📁 ĐÃ LƯU TỆP SRT VÀO GOOGLE DRIVE: {raw_srt_path}")
    print("👉 Bạn hãy mở Google Drive lên, lấy file SRT này về dịch sang tiếng Việt rồi chạy tiếp Giai đoạn 2 bên dưới!")
    return raw_srt_path

exported_raw_srt = run_ocr_and_export_srt(LOCAL_VIDEO_PATH, HARDCODED_CONFIG['ocr_roi'])

---
## 🎯 GIAI ĐOẠN 2: LỒNG TIẾNG TTS (GPU) & RENDER VIDEO LƯU VỀ GOOGLE DRIVE
> Sau khi bạn đã dịch xong tệp SRT sang tiếng Việt (giữ nguyên mốc thời gian):
> 1. Tải tệp SRT đã dịch lên thư mục `vTranslator_Outputs` trên Google Drive của bạn (ví dụ đặt tên là `ten_video_vi.srt`).
> 2. Chạy 2 ô code bên dưới (Ô 5 & Ô 6) để Colab tự động sinh giọng đọc VieNeu GPU và Render video hoàn chỉnh!

In [ ]:
# @title 🎙️ 5. GIAI ĐOẠN 2 - BƯỚC 1: Đọc SRT tiếng Việt & Sinh giọng đọc VieNeu (GPU Puck)
import numpy as np
import soundfile as sf
import torch
from vieneu.v3turbo import V3TurboVieNeuTTS

# 1. Xác định tệp SRT tiếng Việt
base_name = os.path.splitext(os.path.basename(INPUT_VIDEO_SOURCE))[0]
target_srt = TRANSLATED_SRT_PATH.strip()
if not target_srt:
    candidate_1 = os.path.join(OUTPUT_DRIVE_DIR, f"{base_name}_vi.srt")
    candidate_2 = os.path.join(OUTPUT_DRIVE_DIR, f"{base_name}_goc.srt")
    if os.path.exists(candidate_1):
        target_srt = candidate_1
    elif os.path.exists(candidate_2):
        print(f"⚠️ Chưa tìm thấy file '{base_name}_vi.srt', đang tạm dùng '{base_name}_goc.srt' để test.")
        target_srt = candidate_2
    else:
        raise FileNotFoundError(f"Không tìm thấy tệp SRT tại {candidate_1}")

print(f"📄 Đang đọc tệp phụ đề tiếng Việt từ Drive: {target_srt}")

# Hàm phân tích cú pháp SRT
def parse_srt_file(srt_path):
    with open(srt_path, "r", encoding="utf-8-sig") as f:
        content = f.read()
    blocks = re.split(r'\n\s*\n', content.strip())
    parsed = []
    for idx, block in enumerate(blocks, 1):
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if not lines:
            continue
        t_idx = 1 if lines[0].isdigit() and len(lines) > 1 else 0
        if t_idx >= len(lines):
            continue
        m = re.search(r'(\d+):(\d+):(\d+)[,\.](\d+)\s*-->\s*(\d+):(\d+):(\d+)[,\.](\d+)', lines[t_idx])
        if not m:
            continue
        h1, m1, s1, ms1 = map(int, m.groups()[:4])
        h2, m2, s2, ms2 = map(int, m.groups()[4:])
        s_ms = h1*3600000 + m1*60000 + s1*1000 + ms1
        e_ms = h2*3600000 + m2*60000 + s2*1000 + ms2
        txt = " ".join(lines[t_idx+1:])
        txt = re.sub(r'<[^>]+>', '', txt).strip()
        if txt:
            parsed.append({
                "segment_id": f"seg_{idx:04d}",
                "start_ms": s_ms,
                "end_ms": e_ms,
                "translated_text": txt
            })
    return parsed

translated_segments = parse_srt_file(target_srt)
print(f"✅ Đã tải {len(translated_segments)} câu phụ đề tiếng Việt.")

# 2. Khởi tạo VieNeu TTS GPU
TEMP_AUDIO_DIR = "/content/temp_tts"
os.makedirs(TEMP_AUDIO_DIR, exist_ok=True)
MODEL_DIR = "/content/vieneu_models"
if not os.path.exists(MODEL_DIR):
    print("Đang tải mô hình VieNeu-TTS v3 Turbo (~640 MB)...\n")
    !git clone --depth 1 https://huggingface.co/pnnbao-ump/VieNeu-TTS-v3-Turbo {MODEL_DIR}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Khởi tạo VieNeu trên thiết bị: {device.upper()}")

tts_engine = V3TurboVieNeuTTS(
    backbone_repo=MODEL_DIR,
    backend="onnx",
    device=device,
    precision="fp16" if device == "cuda" else "fp32",
    onnx_subfolder="onnx_update",
    onnx_dir=os.path.join(MODEL_DIR, "onnx_update")
)

# Đăng ký mẫu giọng Gemini Puck
PUCK_PRESET_DATA = json.loads('''{"description": "Nam \u00b7 VieNeu On-Device (Gemini Clone) \u00b7 \u1ea4m \u00e1p, t\u1ef1 nhi\u00ean, sinh \u0111\u1ed9ng", "gender": "male", "style": "tu_nhien", "speaker_emb": [-1.324688, 0.745515, -0.083313, -0.162971, 0.477377, -0.263665, -0.947508, -2.033532, -0.509122, 1.083447, -1.02818, 0.676652, 0.479334, 0.756048, 1.604676, -0.637126, 1.00832, 0.485266, -1.873078, 0.689014, -1.699748, 1.594376, -0.749535, -0.167668, -1.141941, -0.035546, -1.01793, 1.443447, -1.01775, 0.14532, -0.48197, -1.236021, -1.528414, -0.002567, 0.499717, -2.393111, -0.218168, -0.23934, -0.848525, 1.797361, 1.639783, -0.657792, -0.294197, -2.305489, -1.20832, -1.116152, 1.010739, -0.714208, 1.241649, -0.616038, 0.460485, 0.006641, -0.065992, -0.400755, 0.69954, -0.600216, 0.943077, -1.413405, -1.642275, -0.179906, -0.332723, 1.945948, 0.863598, -0.130204, -0.767459, 1.856682, -1.0881, 0.286101, 0.115818, -0.454092, -2.096242, 2.565349, -0.02556, -0.050284, -0.955268, -0.46618, -0.097082, 0.120214, -0.867111, 1.117914, 0.209531, 1.338386, -0.926472, 0.866153, 1.038154, 0.247976, 1.222646, 0.176234, -2.136974, -1.647753, -0.211337, 0.500036, -0.009386, -0.291441, -0.049974, -1.254627, 1.901904, -0.00801, 0.457748, 0.617437, 0.781578, -1.34431, -0.303509, 1.026389, 0.287601, -0.962893, 0.689803, 0.644364, -0.738969, -0.423707, -2.279489, 0.38158, -2.928731, -0.722206, -0.947059, 1.381665, -0.692702, -0.234719, -0.863875, 0.615018, 0.054892, -0.359565, 0.098177, -1.279572, -1.166363, -0.12619, -0.699403, -0.107034, -0.266925, -0.743258, -2.063514, -0.632155, -0.524736, 1.240924, 0.377499, 0.96363, 0.816053, -1.559094, -1.123383, 0.088264, 0.816568, -0.456721, 0.312132, 0.328237, 0.334077, 0.780122, 0.977866, -0.28108, 0.252057, -1.433709, -2.629877, -0.214224, 0.5275, 0.359448, 1.243578, 0.606692, -0.610098, -1.473602, 0.396662, -0.771317, -1.228427, -1.253356, 1.838266, -3.131353, 1.071196, 1.265326, 0.619322, -0.843278, -1.420022, -1.90344, -1.026617, 0.095406, 0.259419, -0.833887, 0.771662, -0.942599, 0.789813, 1.299518, -0.292943, -0.598698, -1.463207, -0.911031, 0.428798, -1.207538, 0.224907, 1.039882, 0.262953, -1.042146, -0.68609, -0.406709, -0.268527, 0.237608], "codes": [[642, 10, 725, 283, 90, 811, 317, 196, 137, 886, 170, 521, 638, 393, 786, 965], [642, 841, 21, 555, 950, 255, 793, 737, 813, 891, 352, 63, 259, 212, 379, 100], [947, 494, 509, 794, 42, 458, 1013, 909, 194, 103, 208, 504, 816, 133, 713, 375], [586, 996, 331, 846, 582, 254, 186, 188, 753, 441, 520, 130, 479, 856, 172, 366], [280, 301, 710, 738, 715, 253, 616, 504, 905, 17, 491, 591, 934, 14, 479, 596], [851, 256, 12, 491, 471, 495, 678, 983, 854, 958, 809, 445, 428, 668, 868, 8], [209, 996, 938, 396, 294, 433, 375, 547, 896, 899, 33, 11, 228, 36, 604, 644], [840, 4, 889, 325, 548, 313, 15, 560, 212, 524, 777, 88, 139, 483, 890, 504], [927, 73, 664, 197, 923, 767, 223, 416, 766, 716, 902, 326, 507, 259, 109, 739], [303, 442, 446, 747, 600, 60, 796, 786, 914, 774, 202, 280, 766, 814, 454, 30], [560, 587, 769, 382, 673, 289, 228, 365, 669, 74, 153, 370, 938, 19, 245, 52], [862, 254, 499, 365, 547, 667, 310, 139, 262, 658, 723, 416, 75, 24, 699, 259], [365, 254, 999, 992, 808, 356, 36, 359, 307, 290, 242, 262, 403, 901, 321, 929], [642, 841, 200, 3, 535, 776, 839, 659, 687, 955, 170, 717, 43, 423, 769, 373], [825, 288, 375, 624, 226, 117, 268, 327, 784, 279, 983, 190, 118, 489, 289, 112], [465, 485, 786, 217, 368, 152, 201, 292, 965, 633, 688, 239, 111, 903, 962, 1020], [685, 538, 753, 276, 870, 680, 580, 475, 1017, 442, 906, 754, 42, 897, 199, 19], [780, 572, 155, 116, 705, 905, 411, 500, 618, 693, 873, 950, 87, 189, 89, 41], [764, 170, 689, 491, 788, 111, 446, 980, 782, 754, 502, 649, 636, 753, 764, 269], [244, 59, 438, 725, 149, 136, 946, 525, 961, 883, 126, 635, 160, 1022, 749, 772], [541, 71, 89, 145, 643, 331, 189, 510, 505, 144, 206, 566, 118, 99, 317, 673], [373, 986, 368, 560, 423, 998, 500, 710, 973, 835, 115, 451, 454, 71, 986, 87], [986, 901, 380, 297, 240, 426, 511, 128, 997, 885, 672, 352, 152, 672, 188, 250], [866, 541, 147, 122, 694, 21, 962, 707, 574, 838, 72, 421, 60, 269, 756, 612], [990, 541, 147, 986, 66, 241, 644, 46, 823, 148, 114, 797, 874, 904, 85, 555], [717, 335, 759, 43, 603, 342, 626, 117, 915, 449, 611, 824, 400, 256, 74, 41], [402, 494, 478, 367, 201, 404, 988, 196, 102, 606, 433, 591, 438, 894, 652, 848], [92, 419, 888, 788, 126, 66, 4, 149, 447, 272, 271, 373, 459, 344, 130, 843], [228, 42, 847, 636, 579, 980, 757, 88, 951, 637, 729, 490, 730, 622, 227, 529], [651, 458, 669, 266, 756, 765, 905, 938, 365, 537, 472, 695, 899, 851, 1003, 16], [3, 167, 142, 235, 342, 1002, 42, 518, 550, 115, 129, 865, 722, 29, 65, 320], [832, 424, 148, 205, 934, 916, 74, 471, 106, 850, 796, 720, 413, 531, 514, 998], [402, 490, 807, 197, 469, 302, 283, 457, 751, 745, 511, 318, 489, 352, 61, 766], [767, 787, 596, 118, 827, 110, 810, 660, 150, 25, 855, 342, 848, 410, 431, 781], [933, 374, 66, 909, 308, 441, 290, 174, 958, 417, 503, 107, 718, 909, 997, 943], [699, 48, 138, 853, 615, 13, 586, 932, 244, 15, 879, 969, 496, 492, 506, 481], [717, 157, 291, 799, 739, 673, 385, 912, 205, 737, 491, 953, 635, 904, 315, 625], [876, 54, 344, 773, 561, 874, 41, 434, 31, 787, 316, 18, 108, 773, 302, 86], [825, 443, 791, 483, 242, 330, 516, 297, 564, 746, 577, 540, 779, 654, 809, 960], [993, 738, 827, 832, 133, 875, 73, 261, 489, 302, 769, 692, 177, 891, 482, 33], [482, 419, 769, 968, 252, 557, 793, 882, 999, 293, 255, 360, 943, 742, 275, 712]]}''')
tts_engine._preset_voices["Gemini Puck"] = {
    "description": PUCK_PRESET_DATA.get("description", "Nam Gemini Puck"),
    "gender": PUCK_PRESET_DATA.get("gender", "male"),
    "speaker_emb": np.asarray(PUCK_PRESET_DATA["speaker_emb"], dtype=np.float32),
    "codes": np.asarray(PUCK_PRESET_DATA["codes"], dtype=np.int64)
}
print("✅ Đã nạp giọng 'Gemini Puck' thành công!")

# 3. Sinh toàn bộ file audio lồng tiếng
total_segs = len(translated_segments)
print(f"Bắt đầu sinh âm thanh cho {total_segs} câu bằng GPU...\n")
t_tts = time.time()

for idx, s in enumerate(translated_segments):
    txt = s['translated_text'].strip()
    out_wav = os.path.join(TEMP_AUDIO_DIR, f"{s['segment_id']}.wav")
    if not os.path.exists(out_wav):
        try:
            audio = tts_engine.infer(txt, voice="Gemini Puck", temperature=0.82)
            tts_engine.save(audio, out_wav)
            s['tts_audio_path'] = out_wav
        except Exception as e:
            print(f"Lỗi sinh câu {s['segment_id']}: {e}")
    else:
        s['tts_audio_path'] = out_wav
        
    if (idx + 1) % 100 == 0 or (idx + 1) == total_segs:
        print(f"⏳ Tiến độ TTS: {idx+1}/{total_segs} câu ({time.time() - t_tts:.1f}s)")

print("\n✅ Đã hoàn thành sinh toàn bộ giọng đọc TTS!")

In [ ]:
# @title 🎬 6. GIAI ĐOẠN 2 - BƯỚC 2: Ghép Audio & Render Video Hoàn Chỉnh Lưu Vào Google Drive
import subprocess

# 1. Đo thời lượng video
probe_cmd = f"ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 '{LOCAL_VIDEO_PATH}'"
total_duration_sec = float(subprocess.check_output(probe_cmd, shell=True).decode().strip())
sample_rate = 44100
total_samples = int(total_duration_sec * sample_rate) + sample_rate * 5
master_audio = np.zeros(total_samples, dtype=np.float32)

print(f"Tổng thời lượng: {total_duration_sec:.1f}s ({total_duration_sec/60:.1f} phút)")
print("Đang căn timeline rãnh thuyết minh...")

for s in translated_segments:
    wav_path = s.get('tts_audio_path')
    if wav_path and os.path.exists(wav_path):
        data, sr = sf.read(wav_path, dtype='float32')
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        start_sample = int((s['start_ms'] / 1000.0) * sample_rate)
        end_sample = min(total_samples, start_sample + len(data))
        dur_samples = end_sample - start_sample
        if dur_samples > 0:
            master_audio[start_sample:end_sample] += data[:dur_samples]

master_wav_path = "/content/master_dubbing.wav"
sf.write(master_wav_path, master_audio, sample_rate, subtype='PCM_16')
print("✅ Đã tạo Master Dubbing Audio!")

# 2. Sinh phụ đề ASS
ass_sub_path = "/content/subtitles.ass"
style = HARDCODED_CONFIG['subtitle_style']
ass_header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{style['font_name']},{style['font_size']},{style['primary_color']},&H000000FF,{style['outline_color']},&H00000000,1,0,0,0,100,100,0,0,1,{style['outline_width']},0,{style['alignment']},20,20,{style['margin_v']},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""

def ms_to_ass(ms):
    h = ms // 3600000
    m = (ms % 3600000) // 60000
    s = (ms % 60000) // 1000
    cs = (ms % 1000) // 10
    return f"{h}:{m:02d}:{s:02d}.{cs:02d}"

with open(ass_sub_path, 'w', encoding='utf-8') as f:
    f.write(ass_header)
    for s in translated_segments:
        t_vi = s['translated_text'].strip()
        if t_vi:
            f.write(f"Dialogue: 0,{ms_to_ass(s['start_ms'])},{ms_to_ass(s['end_ms'])},Default,,0,0,0,,{t_vi}\n")

print("✅ Đã tạo tệp phụ đề ASS chuẩn hóa!")

# 3. Thiết lập FFmpeg Filter Graph chuẩn của vTranslator
base_name = os.path.splitext(os.path.basename(INPUT_VIDEO_SOURCE))[0]
final_output_path = os.path.join(OUTPUT_DRIVE_DIR, f"{base_name}_vi.mp4")

# Blur box
b = HARDCODED_CONFIG['blur']
bx, by, bw, bh = b['x'], b['y'], b['width'], b['height']
brad = b['blur_radius']

filter_parts = []
# Thứ tự chuẩn: 1. Làm mờ trước -> 2. Lật gương ngang (hflip) -> 3. Chèn Logo -> 4. Burn ASS
filter_parts.append(f"[0:v]split[m_blur][p_blur];[p_blur]crop=w={bw}:h={bh}:x={bx}:y={by}[pc_blur];[pc_blur]avgblur=sizeX={brad}:sizeY={brad}[pb_blur];[m_blur][pb_blur]overlay=x={bx}:y={by}[v_blurred]")
filter_parts.append("[v_blurred]hflip[v_flipped]")
curr_v = "[v_flipped]"

inputs = ["-i", LOCAL_VIDEO_PATH, "-i", master_wav_path]

if LOCAL_LOGO_PATH and os.path.exists(LOCAL_LOGO_PATH):
    inputs.extend(["-i", LOCAL_LOGO_PATH])
    l_scale = HARDCODED_CONFIG['logo']['scale_percent'] / 100.0
    l_op = HARDCODED_CONFIG['logo']['opacity']
    filter_parts.append(f"[2:v]scale=iw*{l_scale}:-1,format=rgba,colorchannelmixer=aa={l_op}[logo_scaled];{curr_v}[logo_scaled]overlay=main_w-overlay_w-24:24[v_with_logo]")
    curr_v = "[v_with_logo]"

escaped_ass = ass_sub_path.replace("\\", "/").replace(":", "\\:")
filter_parts.append(f"{curr_v}subtitles='{escaped_ass}'[v_final]")

# Audio mix: 17% gốc + 100% lồng tiếng
orig_vol = HARDCODED_CONFIG['audio']['original_volume']
dub_vol = HARDCODED_CONFIG['audio']['dubbing_volume']
audio_filter = f"[0:a]volume={orig_vol:.2f},aresample=44100,aformat=sample_fmts=fltp:channel_layouts=stereo[a_orig];[1:a]volume={dub_vol:.2f},aresample=44100,aformat=sample_fmts=fltp:channel_layouts=stereo[a_dub];[a_orig][a_dub]amix=inputs=2:duration=longest:normalize=0[a_final]"

full_filter = ";".join(filter_parts) + ";" + audio_filter

# Render lệnh FFmpeg với NVENC GPU
render_cmd = [
    "ffmpeg", "-y",
    *inputs,
    "-filter_complex", full_filter,
    "-map", "[v_final]",
    "-map", "[a_final]",
    "-c:v", "h264_nvenc",
    "-preset", "p4",
    "-c:a", "aac", "-b:a", "192k",
    "-movflags", "+faststart",
    final_output_path
]

print(f"\n🚀 Bắt đầu Render video bằng GPU NVIDIA (Lưu trực tiếp vào Google Drive)...")
print(f"📂 Đích lưu Drive: {final_output_path}")
t_render = time.time()
res = subprocess.run(render_cmd, capture_output=True, text=True)

if res.returncode == 0:
    print(f"\n🎉🎉 XUẤT SẮC! Video đã render xong và LƯU TRỰC TIẾP VÀO GOOGLE DRIVE trong {time.time() - t_render:.1f}s!")
    print(f"📁 File trên Google Drive: {final_output_path}")
else:
    print("Chuyển sang bộ mã hóa CPU rất nhanh...")
    render_cmd[render_cmd.index("-c:v") + 1] = "libx264"
    render_cmd[render_cmd.index("-preset") + 1] = "veryfast"
    subprocess.run(render_cmd, check=True)
    print(f"\n🎉🎉 XUẤT SẮC! Video đã render xong và lưu vào: {final_output_path}")